
# EmotionLab
## Análisis de datos y Machine Learning

**Proyecto Capstone — Ingeniería en Informática**

**Integrantes:**
- Lorena Calfin
- Martín Carrere
- Emily Gonzales

**Profesor guía:** Cristian Molina

---

## Objetivo general

Desarrollar un módulo inteligente para analizar las
respuestas emocionales autodeclaradas de estudiantes
durante presentaciones orales, identificar patrones
mediante técnicas de Machine Learning y proporcionar
una clasificación exploratoria de sus niveles de
tensión percibida.

## Descripción

EmotionLab recopila respuestas emocionales autodeclaradas mediante tres formularios aplicados durante una experiencia de presentación oral: antes de presentar, después de la modalidad fácil y después de la modalidad difícil.

El módulo de análisis utilizará la información de los tres formularios para desarrollar un análisis global del estudiante, identificando patrones mediante técnicas de Machine Learning y proporcionando una clasificación exploratoria de su nivel general de tensión percibida.

Adicionalmente, se analizará la evolución de las respuestas entre etapas, permitiendo visualizar los cambios en el nerviosismo y el estado emocional durante la experiencia.

## Alcance del modelo

Los resultados representarán patrones y niveles
exploratorios de tensión emocional autodeclarada.

No se utilizarán como diagnóstico psicológico ni como
medición clínica de estrés.



# Etapa 1: Carga y revisión inicial del dataset

## Objetivo

En esta etapa trabajaremos con el dataset híbrido que
construimos previamente para el proyecto EmotionLab.

El dataset está compuesto por 600 registros:

- 27 registros originales obtenidos de estudiantes.
- 573 registros sintéticos generados a partir de los datos reales.

Antes de utilizar técnicas de Machine Learning, debemos
comprobar que los datos se carguen correctamente, que las
variables estén disponibles y que podamos distinguir los
registros reales de los sintéticos.

Para esto utilizaremos Python, pandas y Google Colab.


In [1]:

# Bibliotecas necesarias para usos muy generales o usos únicos
import numpy as np
import pandas as pd
import io
from google.colab import files

# estas bibliotecas utilizaremos para visualizar las versiones de cada biblioteca a utilizar,
# si no las tenemos seteadas desde un inicio, a la hora de hacer el importe a AWS, AWS instalará las versiones mas recientes de estas bibliotecas
# lo cual puede generar conflictos.
import sys
import pandas as pd
import scipy
import sklearn
import matplotlib
import seaborn as sns
import joblib



## 1.1 Configuración general del proyecto

En esta sección definiremos los parámetros que
utilizaremos durante el desarrollo.

Establecer una semilla aleatoria permitirá reproducir
los experimentos que involucren procesos aleatorios.

También definiremos la cantidad esperada de registros
y las variables que identifican cada formulario.

El análisis de las respuestas observadas se realizará
sobre los participantes reales. Los registros sintéticos
se conservarán para experimentación y pruebas técnicas,
sin considerarlos nuevas observaciones independientes.


In [2]:

# Definimos una semilla para que los experimentos
# aleatorios puedan reproducirse.
# La semilla se crea con el número 42 sin algún proposito en especifico, ya que puede ser cualquier número y el resultado será el mismo, mientras el número de la semilla sea el mismo

SEMILLA = 42
np.random.seed(SEMILLA)

# Cantidades esperadas en el dataset.
# Las comprobaremos al cargar el archivo.

REGISTROS_REALES = 27
REGISTROS_SINTETICOS = 573
REGISTROS_TOTALES = 600

# Valores permitidos por los formularios.

ESCALA_MINIMA = 1
ESCALA_MAXIMA = 9

# Identificamos las variables de cada etapa.

variables_iniciales = [
    "estado_pre",
    "nerviosismo_pre",
    "preocupacion_pre",
    "preparacion_pre"
]

variables_facil = [
    "estado_facil",
    "nerviosismo_facil",
    "satisfaccion_facil",
    "alivio_facil"
]

variables_dificil = [
    "estado_dificil",
    "nerviosismo_dificil",
    "satisfaccion_dificil",
    "alivio_dificil"
]

# Unimos las variables para facilitar
# su utilización en los análisis posteriores.

variables_cuestionario = (
    variables_iniciales
    + variables_facil
    + variables_dificil
)

print("Semilla aleatoria:", SEMILLA)
print("Registros esperados:", REGISTROS_TOTALES)
print("Variables del cuestionario:", len(variables_cuestionario))
print("Escala:", ESCALA_MINIMA, "a", ESCALA_MAXIMA)


Semilla aleatoria: 42
Registros esperados: 600
Variables del cuestionario: 12
Escala: 1 a 9



## 1.2 Verificación de versiones

Antes de comenzar el procesamiento, registraremos
las versiones de Python y de las principales
bibliotecas utilizadas.

Esto permitirá documentar el entorno de desarrollo
y facilitará la reproducción de los experimentos.

También será útil al trasladar los modelos entrenados
desde Google Colab al backend de AWS, donde necesitaremos
mantener un entorno compatible.


In [3]:

# Consultamos las versiones de las herramientas
# principales del proyecto.

versiones = {
    "Python": sys.version.split()[0],
    "pandas": pd.__version__,
    "NumPy": np.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
    "Seaborn": sns.__version__,
    "scikit-learn": sklearn.__version__,
    "joblib": joblib.__version__
}

# Organizamos las versiones en una tabla
# para facilitar su lectura.

tabla_versiones = pd.DataFrame(
    versiones.items(),
    columns=["Herramienta", "Versión instalada"]
)

print("Versiones del entorno de desarrollo:")
display(tabla_versiones)


Versiones del entorno de desarrollo:


,Herramienta,Versión instalada
0,Python,3.13.18
1,pandas,2.2.3
2,NumPy,2.4.6
3,SciPy,1.16.3
4,Matplotlib,3.10.0
5,Seaborn,0.13.2
6,scikit-learn,1.6.1
7,joblib,1.6.0



## Conclusión de la Etapa 1

Se configuró el entorno de desarrollo para el módulo
de análisis y Machine Learning de EmotionLab.

Se importaron las bibliotecas necesarias para el
procesamiento de datos, análisis estadístico,
visualización, entrenamiento no supervisado y
evaluación de agrupaciones.

También se establecieron los parámetros generales
del proyecto y se identificaron las doce variables
correspondientes a los tres formularios.

Finalmente, se incorporó un procedimiento para
registrar las versiones de las herramientas utilizadas.

Con esta configuración, el entorno queda preparado
para cargar y validar el dataset de EmotionLab.



# Etapa 2: Carga y validación del dataset

## Objetivo

Cargar el dataset híbrido construido previamente
para EmotionLab y comprobar que su estructura
sea adecuada para las siguientes etapas del proyecto.

El conjunto de datos contiene 600 registros:

- 27 registros originales de estudiantes.
- 573 registros sintéticos generados experimentalmente.

Durante esta etapa verificaremos la integridad de
la información, la disponibilidad de las variables
y la procedencia de los registros.

## Procedimiento

1. Cargar el archivo CSV del dataset.
2. Revisar su estructura y tipos de datos.
3. Comprobar la cantidad y procedencia de registros.
4. Examinar la disponibilidad de respuestas.
5. Validar que las respuestas respeten la escala
   definida en los formularios.
6. Separar los datos reales de los sintéticos.

## Consideraciones

No eliminaremos ni modificaremos respuestas originales.

Los valores ausentes se conservarán para estudiar
su tratamiento durante el preprocesamiento.

El dataset sintético se utilizará principalmente
para experimentación y pruebas técnicas, mientras
que las conclusiones sobre estudiantes se basarán
en los registros reales disponibles.


#2.1 Carga del dataset hibrido

In [4]:

# Cargamos el dataset generado anteriormente.
print("Selecciona el dataset de EmotionLab.")

archivo_subido = files.upload()

# Comprobamos que se haya seleccionado un solo archivo.
if len(archivo_subido) != 1:
    raise ValueError("Debes cargar solamente un archivo.")

nombre_archivo = list(archivo_subido.keys())[0]
contenido = archivo_subido[nombre_archivo]

# Leemos el dataset según su formato.
if nombre_archivo.lower().endswith(".csv"):
    dataset = pd.read_csv(io.BytesIO(contenido), encoding="utf-8-sig")
elif nombre_archivo.lower().endswith(".xlsx"):
    dataset = pd.read_excel(io.BytesIO(contenido))
else:
    raise ValueError("El archivo debe ser CSV o XLSX.")

print("Archivo cargado:", nombre_archivo)
print("Dimensiones:", dataset.shape)


Selecciona el dataset de EmotionLab.


Saving EmotionLab_Dataset_Hibrido_600.csv to EmotionLab_Dataset_Hibrido_600.csv
Archivo cargado: EmotionLab_Dataset_Hibrido_600.csv
Dimensiones: (600, 21)



## 2.2 Revisión inicial de la estructura

Antes de procesar los datos, revisaremos cómo están
organizados dentro del archivo.

Para ello utilizaremos las funciones de pandas:

- shape: identifica la cantidad de filas y columnas.
- head: muestra una vista preliminar del dataset.
- info: presenta los tipos de datos y valores presentes.
- dtypes: permite revisar el tipo de cada variable.

Esta revisión es importante para detectar posibles
problemas de lectura o inconsistencias en los datos.

También permitirá comprobar que las respuestas de
los formularios se encuentren organizadas correctamente.


In [5]:

# Revisamos el tamaño del dataset.
print("Registros:", dataset.shape[0])
print("Columnas:", dataset.shape[1])

# Visualizamos los primeros cinco registros.
print("\nPrimeras filas:")
display(dataset.head())

# Consultamos los tipos de datos de las columnas.
print("\nTipos de datos:")
display(dataset.dtypes.to_frame(name="Tipo de dato"))

# Revisamos la estructura general y los valores no nulos.
print("\nInformación general:")
dataset.info()


Registros: 600
Columnas: 21

Primeras filas:


,id_estudiante,tipo_origen,bloque_origen,edad,genero,n_respuestas,integridad,alivio_dificil,alivio_facil,estado_dificil,...,estado_pre,nerviosismo_dificil,nerviosismo_facil,nerviosismo_pre,preocupacion_pre,preparacion_pre,satisfaccion_dificil,satisfaccion_facil,tipo_registro,id_semilla
0,EM_001,nombre,1,17.0,m,12,completo,1.0,3.0,4.0,...,1.0,3.0,4.0,3.0,4.0,3.0,1.0,2.0,real,NaN
1,EM_002,nombre,1,17.0,m,12,completo,5.0,2.0,2.0,...,2.0,3.0,3.0,3.0,1.0,4.0,4.0,1.0,real,NaN
2,EM_003,nombre,1,17.0,m,12,completo,1.0,1.0,4.0,...,1.0,3.0,1.0,1.0,9.0,1.0,2.0,1.0,real,NaN
3,EM_004,nombre,1,18.0,f,12,completo,1.0,1.0,2.0,...,3.0,2.0,1.0,5.0,4.0,3.0,1.0,2.0,real,NaN
4,EM_005,nombre,1,18.0,f,12,completo,3.0,6.0,2.0,...,2.0,3.0,6.0,3.0,1.0,4.0,1.0,7.0,real,NaN



Tipos de datos:


,Tipo de dato
id_estudiante,object
tipo_origen,object
bloque_origen,int64
edad,float64
genero,object
n_respuestas,int64
integridad,object
alivio_dificil,float64
alivio_facil,float64
estado_dificil,float64



Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_estudiante         600 non-null    object 
 1   tipo_origen           600 non-null    object 
 2   bloque_origen         600 non-null    int64  
 3   edad                  598 non-null    float64
 4   genero                571 non-null    object 
 5   n_respuestas          600 non-null    int64  
 6   integridad            600 non-null    object 
 7   alivio_dificil        349 non-null    float64
 8   alivio_facil          382 non-null    float64
 9   estado_dificil        349 non-null    float64
 10  estado_facil          418 non-null    float64
 11  estado_pre            595 non-null    float64
 12  nerviosismo_dificil   349 non-null    float64
 13  nerviosismo_facil     418 non-null    float64
 14  nerviosismo_pre       571 non-null    float64
 15  p


## 2.3 Verificación de registros y variables

El dataset fue construido combinando las respuestas
reales de los participantes con registros sintéticos.

Necesitamos verificar que se mantenga la cantidad
esperada de registros y que sus identificadores
no se encuentren duplicados.

También comprobaremos que estén presentes las
12 variables correspondientes a los formularios:

- Evaluación inicial.
- Evaluación posterior a la dificultad fácil.
- Evaluación posterior a la dificultad difícil.

Estas variables serán utilizadas para estudiar
los patrones de tensión emocional percibida.


In [6]:

# Definimos las columnas que debe contener el dataset.
columnas_necesarias = [
    "id_estudiante", "tipo_registro",
    "id_semilla", "n_respuestas"
] + variables_cuestionario

# Revisamos si falta alguna columna.
columnas_faltantes = [
    columna for columna in columnas_necesarias
    if columna not in dataset.columns
]

if columnas_faltantes:
    raise ValueError(f"Faltan columnas: {columnas_faltantes}")

# Comprobamos la cantidad de registros.
if len(dataset) != REGISTROS_TOTALES:
    raise ValueError("La cantidad de registros no coincide.")

print("Estructura del dataset verificada.")


Estructura del dataset verificada.


In [7]:

#  Comprobamos que cada registro tenga un identificador único.
if dataset["id_estudiante"].isna().any():
    raise ValueError("Existen registros sin identificador.")

if not dataset["id_estudiante"].is_unique:
    raise ValueError("Se encontraron identificadores duplicados.")

# Contamos los registros reales y sintéticos.
conteo_registros = dataset["tipo_registro"].value_counts()

if conteo_registros.get("real", 0) != REGISTROS_REALES:
    raise ValueError("La cantidad de registros reales no coincide.")

if conteo_registros.get("sintetico_exploratorio", 0) != REGISTROS_SINTETICOS:
    raise ValueError("La cantidad de registros sintéticos no coincide.")

print("Validación de registros completada.")
display(conteo_registros.to_frame(name="Registros"))


Validación de registros completada.


,Registros
tipo_registro,
sintetico_exploratorio,573
real,27



## 2.4 Disponibilidad y completitud de respuestas

En la recopilación de datos no todos los estudiantes
completaron los tres formularios.

Por esta razón, antes de aplicar Machine Learning
debemos conocer cuántas respuestas existen y
cuántas se encuentran ausentes.

Revisaremos:

- Total de respuestas presentes.
- Total de respuestas ausentes.
- Disponibilidad de cada variable.
- Cantidad de formularios completos e incompletos.

No reemplazaremos los valores ausentes en esta etapa.

Su tratamiento se definirá posteriormente, durante
la preparación de los datos para Machine Learning.


In [8]:

# Contamos las respuestas presentes y ausentes por pregunta.
respuestas_presentes = dataset[variables_cuestionario].notna().sum()
respuestas_ausentes = dataset[variables_cuestionario].isna().sum()

# Creamos una tabla con la disponibilidad de cada variable.
resumen_respuestas = pd.DataFrame({
    "Presentes": respuestas_presentes,
    "Ausentes": respuestas_ausentes
})

resumen_respuestas["Disponibilidad (%)"] = (
    resumen_respuestas["Presentes"] / len(dataset) * 100
).round(2)

print("Disponibilidad de respuestas por variable:")
display(resumen_respuestas)

# Calculamos los totales del cuestionario.
total_presentes = int(respuestas_presentes.sum())
total_ausentes = int(respuestas_ausentes.sum())

print("Respuestas presentes:", total_presentes)
print("Respuestas ausentes:", total_ausentes)
print("Respuestas posibles:", len(dataset) * len(variables_cuestionario))


Disponibilidad de respuestas por variable:


,Presentes,Ausentes,Disponibilidad (%)
estado_pre,595,5,99.17
nerviosismo_pre,571,29,95.17
preocupacion_pre,571,29,95.17
preparacion_pre,571,29,95.17
estado_facil,418,182,69.67
nerviosismo_facil,418,182,69.67
satisfaccion_facil,418,182,69.67
alivio_facil,382,218,63.67
estado_dificil,349,251,58.17
nerviosismo_dificil,349,251,58.17


Respuestas presentes: 5340
Respuestas ausentes: 1860
Respuestas posibles: 7200



## 2.5 Validación de la escala de evaluación

Los formularios de EmotionLab utilizan una escala
ordinal de 1 a 9, donde el valor 1 representa
la evaluación más favorable y el valor 9
la menos favorable.

Antes de realizar cálculos estadísticos debemos
asegurarnos de que las respuestas presentes
correspondan a valores enteros dentro de ese rango.

Para ello identificaremos:

- Respuestas menores que 1.
- Respuestas mayores que 9.
- Respuestas decimales.
- Valores no numéricos.

Los campos vacíos no se considerarán respuestas
inválidas, ya que representan información ausente.

En caso de encontrar valores incorrectos,
los revisaremos antes de modificar el dataset.


In [9]:

# Organizamos las respuestas para revisar cada valor individualmente.
revision_escala = dataset[
    ["id_estudiante"] + variables_cuestionario
].melt(id_vars="id_estudiante", var_name="variable", value_name="respuesta")

# Convertimos las respuestas a números cuando sea posible.
revision_escala["valor_numerico"] = pd.to_numeric(
    revision_escala["respuesta"], errors="coerce"
)

# Detectamos respuestas que no son numéricas.
numeros = revision_escala["valor_numerico"]
no_numericos = revision_escala["respuesta"].notna() & numeros.isna()

# Comprobamos que los valores sean enteros entre 1 y 9.
fuera_de_escala = numeros.notna() & (
    (numeros < ESCALA_MINIMA) |
    (numeros > ESCALA_MAXIMA) |
    (numeros % 1 != 0)
)

# Guardamos las respuestas que no cumplen las condiciones.
valores_invalidos = revision_escala.loc[
    no_numericos | fuera_de_escala,
    ["id_estudiante", "variable", "respuesta"]
].copy()

print("Valores inválidos encontrados:", len(valores_invalidos))

if not valores_invalidos.empty:
    display(valores_invalidos)
else:
    print("Todas las respuestas presentes respetan la escala.")


Valores inválidos encontrados: 0
Todas las respuestas presentes respetan la escala.



## 2.6 Separación por procedencia

El dataset híbrido contiene dos tipos de registros.

Los registros reales corresponden a las respuestas
obtenidas directamente de estudiantes participantes.

Los registros sintéticos fueron construidos a partir
de perfiles originales mediante un procedimiento
experimental de generación de datos.

Para evitar confundir ambas procedencias,
crearemos dos conjuntos de trabajo independientes.

Los datos reales servirán como base para el análisis
de las observaciones recopiladas y para estudiar
la viabilidad de los algoritmos no supervisados.

Los sintéticos se conservarán para experimentos
de procesamiento y pruebas de funcionamiento.

Esta separación no elimina registros ni altera
el archivo original cargado en memoria.


In [10]:

# =========================================================
# 2.6.1. VERIFICACIÓN DE LA ESTRUCTURA DEL DATASET
# =========================================================

# Antes de separar los registros, comprobamos que el
# dataset tenga las columnas que necesitamos.
#
# tipo_registro: indica si una fila es real o sintética.
# id_estudiante: identifica al estudiante registrado.
# id_semilla: relaciona un registro sintético con su origen.
# n_respuestas: indica cuántas respuestas tiene cada registro.

columnas_necesarias = [
    "tipo_registro",
    "id_estudiante",
    "id_semilla",
    "n_respuestas"
]

# Buscamos si falta alguna de las columnas anteriores.

columnas_faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in dataset.columns
]

# Si falta una columna, detenemos la ejecución.
# Esto evita errores menos claros en las siguientes celdas.

if columnas_faltantes:
    raise ValueError(
        f"Faltan columnas en el dataset: {columnas_faltantes}"
    )

# También verificamos que todos los registros tengan
# una procedencia reconocida por EmotionLab.

tipos_permitidos = ["real", "sintetico_exploratorio"]

if not dataset["tipo_registro"].isin(tipos_permitidos).all():
    raise ValueError(
        "Existen registros con una procedencia no reconocida."
    )

print("Estructura del dataset verificada correctamente.")


Estructura del dataset verificada correctamente.


In [11]:

# =========================================================
# 2.6.2. SEPARACIÓN DE REGISTROS SEGÚN SU PROCEDENCIA
# =========================================================

# En EmotionLab trabajamos con dos tipos de registros:
#
# Reales: respuestas obtenidas de estudiantes.
# Sintéticos: registros generados artificialmente
# para realizar exploraciones complementarias.
#
# Los mantenemos separados para no confundir
# información observada con información generada.

datos_reales = dataset.loc[
    dataset["tipo_registro"] == "real"
].copy()

datos_sinteticos = dataset.loc[
    dataset["tipo_registro"] == "sintetico_exploratorio"
].copy()

# Utilizamos .loc[] para seleccionar las filas según
# una condición y .copy() para crear una copia independiente.
#
# De esta manera, si posteriormente modificamos uno
# de estos conjuntos, no alteramos el dataset original.

print("Registros separados correctamente.")

print("Registros reales:", len(datos_reales))
print("Registros sintéticos:", len(datos_sinteticos))


Registros separados correctamente.
Registros reales: 27
Registros sintéticos: 573


In [12]:

# =========================================================
# 2.6.3. VALIDACIÓN DE LA CANTIDAD DE REGISTROS
# =========================================================

# En una etapa anterior definimos cuántos registros
# reales y sintéticos debería contener nuestro dataset.
#
# Ahora comparamos esas cantidades con los datos
# que acabamos de separar.
#
# Esta verificación permite identificar pérdidas,
# duplicaciones inesperadas o errores de clasificación.

cantidad_reales = len(datos_reales)
cantidad_sinteticos = len(datos_sinteticos)

# Comprobamos los registros reales.

if cantidad_reales != REGISTROS_REALES:
    raise ValueError(
        f"Cantidad de registros reales incorrecta. "
        f"Esperados: {REGISTROS_REALES}. "
        f"Encontrados: {cantidad_reales}."
    )

# Comprobamos los registros sintéticos.

if cantidad_sinteticos != REGISTROS_SINTETICOS:
    raise ValueError(
        f"Cantidad de registros sintéticos incorrecta. "
        f"Esperados: {REGISTROS_SINTETICOS}. "
        f"Encontrados: {cantidad_sinteticos}."
    )

print("Cantidad de registros validada correctamente.")
print("Ambos conjuntos coinciden con las cantidades esperadas.")


Cantidad de registros validada correctamente.
Ambos conjuntos coinciden con las cantidades esperadas.


In [13]:

# =========================================================
# 2.6.4. VERIFICACIÓN DE LA TRAZABILIDAD DE LOS DATOS
# =========================================================

# Los registros sintéticos de EmotionLab se encuentran
# relacionados con registros reales utilizados como origen.
#
# La columna id_semilla permite identificar esa relación.
#
# Verificaremos que cada registro sintético tenga una
# referencia válida a un estudiante real.

# Primero comprobamos que los registros reales
# tengan identificadores disponibles.

if datos_reales["id_estudiante"].isna().any():
    raise ValueError(
        "Existen registros reales sin identificador."
    )

# Luego verificamos que ningún registro sintético
# tenga el identificador de origen vacío.

if datos_sinteticos["id_semilla"].isna().any():
    raise ValueError(
        "Existen registros sintéticos sin identificador de origen."
    )

# Creamos un conjunto con los identificadores reales.
#
# set() permite consultar si un identificador existe
# sin recorrer manualmente todas las filas.

ids_reales = set(datos_reales["id_estudiante"])

# isin() comprueba si cada id_semilla corresponde
# a alguno de los identificadores reales.

origen_valido = datos_sinteticos["id_semilla"].isin(ids_reales)

# all() exige que todas las comprobaciones sean verdaderas.

if not origen_valido.all():
    raise ValueError(
        "Se encontraron registros sintéticos "
        "con identificadores de origen inválidos."
    )

print("Trazabilidad de los registros sintéticos verificada.")
print("Todos los registros sintéticos tienen un origen registrado.")


Trazabilidad de los registros sintéticos verificada.
Todos los registros sintéticos tienen un origen registrado.


In [14]:

# =========================================================
# 5. RESUMEN DE PROCEDENCIA DEL DATASET
# =========================================================

# Finalmente, generamos una tabla para observar
# cómo se distribuyen los datos según su procedencia.
#
# groupby() agrupa las filas por tipo_registro.
#
# agg() calcula dos valores para cada grupo:
#
# registros: cantidad de filas que contiene.
# respuestas_declaradas: suma de los valores
# registrados en la columna n_respuestas.

resumen_procedencia = (
    dataset.groupby("tipo_registro")
    .agg(
        registros=("id_estudiante", "size"),
        respuestas_declaradas=("n_respuestas", "sum")
    )
    .reset_index()
)

# Utilizamos size para contar todas las filas,
# independientemente de si alguna tiene un ID vacío.
#
# reset_index() transforma el resultado agrupado
# en una tabla convencional, más fácil de visualizar.

print("RESUMEN DE PROCEDENCIA - EMOTIONLAB")

display(resumen_procedencia)

# Mostramos también las dimensiones de cada conjunto.
#
# shape devuelve una tupla:
# (cantidad de filas, cantidad de columnas).

print("\nDimensiones de los registros reales:")
print(datos_reales.shape)

print("\nDimensiones de los registros sintéticos:")
print(datos_sinteticos.shape)

print("\nSeparación y validación completadas correctamente.")


RESUMEN DE PROCEDENCIA - EMOTIONLAB


,tipo_registro,registros,respuestas_declaradas
0,real,27,196
1,sintetico_exploratorio,573,5144



Dimensiones de los registros reales:
(27, 21)

Dimensiones de los registros sintéticos:
(573, 21)

Separación y validación completadas correctamente.


# 2.7 PREPARACIÓN DE LAS RESPUESTAS REALES

¿Qué hace este proceso?
En EmotionLab, los estudiantes responden cuestionarios en tres momentos: antes de la actividad (fase inicial), después de la presentación fácil y después de la presentación difícil.

Sin embargo, no todos los estudiantes necesariamente completan todas las preguntas. Por eso, necesitamos comprobar cuánta información tenemos disponible antes de aplicar los análisis.

Este proceso revisa exclusivamente los registros reales, identifica las respuestas ausentes y clasifica cada registro como completo, parcial o sin respuestas. Después, analiza cada formulario por separado para detectar en qué fase falta más información.

Es importante aclarar que esta clasificación no mide el estrés ni la tensión del estudiante. Solamente evalúa la disponibilidad de sus respuestas. Tampoco reemplaza los valores faltantes por datos inventados.

In [15]:

# =========================================================
# 2.7.1. PREPARACIÓN DE LAS RESPUESTAS REALES
# =========================================================

# Para analizar la integridad de los cuestionarios,
# primero necesitamos trabajar con las respuestas
# de los estudiantes reales.
#
# Utilizamos datos_reales, creado anteriormente,
# para evitar mezclar respuestas reales y sintéticas.

# Comprobamos que las preguntas del cuestionario
# estén presentes en nuestro conjunto de datos.

columnas_faltantes = [
    columna
    for columna in variables_cuestionario
    if columna not in datos_reales.columns
]

if columnas_faltantes:
    raise ValueError(
        f"Faltan preguntas del cuestionario: {columnas_faltantes}"
    )

# También comprobamos que la lista de preguntas
# no esté vacía.

if len(variables_cuestionario) == 0:
    raise ValueError(
        "No se han definido preguntas para revisar."
    )

# Seleccionamos las columnas correspondientes
# a las respuestas del cuestionario.
#
# replace() transforma los textos vacíos o
# compuestos únicamente por espacios en NaN.
#
# NaN representa un dato ausente en Pandas.

respuestas_reales = datos_reales[
    variables_cuestionario
].replace(
    r"^\s*$",
    np.nan,
    regex=True
)

# Esta operación prepara una tabla para el análisis.
# No reemplazamos las respuestas ausentes por valores
# artificiales ni modificamos el dataset original.

print("Respuestas reales preparadas correctamente.")
print("Estudiantes revisados:", len(respuestas_reales))
print("Preguntas consideradas:", len(variables_cuestionario))


Respuestas reales preparadas correctamente.
Estudiantes revisados: 27
Preguntas consideradas: 12


In [16]:

# =========================================================
# 2.7.2. CLASIFICACIÓN DE LA INTEGRIDAD
# =========================================================

# notna() identifica los valores que NO están vacíos.
#
# sum(axis=1) cuenta las respuestas disponibles
# horizontalmente, es decir, estudiante por estudiante.

datos_reales["respuestas_disponibles"] = (
    respuestas_reales.notna().sum(axis=1)
)

# Guardamos la cantidad total de preguntas
# para comparar cada registro con ese valor.

total_preguntas = len(variables_cuestionario)


# Creamos una función para clasificar los registros.
#
# Completo: respondió todas las preguntas.
# Sin respuestas: no respondió ninguna.
# Parcial: respondió algunas, pero no todas.
#
# Esta clasificación representa integridad de datos,
# no niveles de estrés ni estados psicológicos.

def clasificar_integridad(cantidad):

    if cantidad == total_preguntas:
        return "Completo"

    elif cantidad == 0:
        return "Sin respuestas"

    else:
        return "Parcial"


# apply() ejecuta nuestra función sobre la cantidad
# de respuestas disponibles de cada estudiante.
#
# El resultado se guarda en una nueva columna
# del conjunto de registros reales.

datos_reales["estado_integridad"] = (
    datos_reales["respuestas_disponibles"]
    .apply(clasificar_integridad)
)

print("Integridad de los registros clasificada.")


Integridad de los registros clasificada.


In [17]:

# =========================================================
# 2.7.3. RESUMEN GENERAL DE INTEGRIDAD
# =========================================================

# value_counts() cuenta cuántos estudiantes
# pertenecen a cada categoría de integridad.

resumen_integridad = (
    datos_reales["estado_integridad"]
    .value_counts()
)

# Organizamos las categorías en un orden fijo.
#
# reindex() también permite mostrar categorías
# con cero estudiantes, si fuera necesario.

resumen_integridad = resumen_integridad.reindex(
    ["Completo", "Parcial", "Sin respuestas"],
    fill_value=0
)

# to_frame() convierte el resumen en una tabla.
# Esto permite presentarlo de forma más ordenada.

tabla_integridad = resumen_integridad.to_frame(
    name="Estudiantes"
)

print("INTEGRIDAD DE LOS CUESTIONARIOS REALES")

display(tabla_integridad)

# Verificamos que la suma de estudiantes
# coincida con el total de registros reales.

if resumen_integridad.sum() != len(datos_reales):
    raise ValueError(
        "El resumen no coincide con la cantidad "
        "de registros reales."
    )

print("\nResumen de integridad verificado.")


INTEGRIDAD DE LOS CUESTIONARIOS REALES


,Estudiantes
estado_integridad,
Completo,13
Parcial,9
Sin respuestas,5



Resumen de integridad verificado.


In [18]:

# =========================================================
# 2.7.4. DISPONIBILIDAD DE RESPUESTAS POR FASE
# =========================================================

# Agrupamos las preguntas según el formulario
# al que pertenecen dentro de EmotionLab.
#
# Estas variables fueron definidas previamente
# durante la organización del cuestionario.

fases = {
    "Inicial": variables_iniciales,
    "Fácil": variables_facil,
    "Difícil": variables_dificil
}

# Creamos una lista vacía donde almacenaremos
# los resultados de cada formulario.

resultados_formularios = []

# Recorremos cada fase y sus preguntas.
#
# items() entrega tanto el nombre de la fase
# como la lista de columnas correspondientes.

for fase, columnas in fases.items():

    # Verificamos que las preguntas de esta fase
    # pertenezcan al cuestionario ya preparado.

    if not columnas:
        raise ValueError(
            f"La fase {fase} no tiene preguntas definidas."
        )

    faltantes = [
        columna
        for columna in columnas
        if columna not in respuestas_reales.columns
    ]

    if faltantes:
        raise ValueError(
            f"La fase {fase} contiene preguntas "
            f"no disponibles: {faltantes}"
        )

    # Contamos las respuestas disponibles
    # de cada estudiante en esta fase.

    cantidad = (
        respuestas_reales[columnas]
        .notna()
        .sum(axis=1)
    )

    # Identificamos a los estudiantes que
    # respondieron todas las preguntas de la fase.

    completos = (
        cantidad == len(columnas)
    ).sum()

    # Identificamos los registros que contienen
    # algunas respuestas, pero no todas.

    parciales = (
        (cantidad > 0) &
        (cantidad < len(columnas))
    ).sum()

    # Identificamos a quienes no tienen ninguna
    # respuesta disponible en esta fase.

    sin_respuestas = (
        cantidad == 0
    ).sum()

    # Contamos las respuestas presentes.
    #
    # Este número corresponde a respuestas,
    # no a la cantidad de estudiantes.

    respuestas_presentes = int(cantidad.sum())

    # Guardamos los resultados del formulario.

    resultados_formularios.append({
        "Formulario": fase,
        "Respuestas presentes": respuestas_presentes,
        "Completos": int(completos),
        "Parciales": int(parciales),
        "Sin respuestas": int(sin_respuestas)
    })

# Convertimos los resultados en un DataFrame
# para visualizar las tres fases en una sola tabla.

resumen_formularios_reales = pd.DataFrame(
    resultados_formularios
)

print("DISPONIBILIDAD DE LOS FORMULARIOS REALES")

display(resumen_formularios_reales)

# Cada fase debe clasificar a todos los estudiantes
# reales en una sola categoría de integridad.

for _, fila in resumen_formularios_reales.iterrows():

    total_clasificados = (
        fila["Completos"]
        + fila["Parciales"]
        + fila["Sin respuestas"]
    )

    if total_clasificados != len(datos_reales):
        raise ValueError(
            f"La fase {fila['Formulario']} "
            "no clasifica correctamente los registros."
        )

print("\nRevisión por fase completada correctamente.")


DISPONIBILIDAD DE LOS FORMULARIOS REALES


,Formulario,Respuestas presentes,Completos,Parciales,Sin respuestas
0,Inicial,85,21,1,5
1,Fácil,59,14,1,12
2,Difícil,52,13,0,14



Revisión por fase completada correctamente.


In [19]:

# =========================================================
# 2.7.5. DISPONIBILIDAD TOTAL DE RESPUESTAS REALES
# =========================================================

# Contamos todas las respuestas disponibles.
#
# notna() identifica las celdas con información.
# El primer sum() cuenta por columna.
# El segundo sum() obtiene el total general.

total_respuestas_reales = int(
    respuestas_reales.notna().sum().sum()
)

# Calculamos cuántas respuestas existirían
# si todos los estudiantes hubieran contestado
# todas las preguntas del cuestionario.

respuestas_posibles = (
    len(datos_reales) * len(variables_cuestionario)
)

# Calculamos cuántas respuestas están ausentes.

respuestas_faltantes = (
    respuestas_posibles - total_respuestas_reales
)

# Calculamos el porcentaje de disponibilidad.
#
# Verificamos que existan respuestas posibles
# para evitar una división por cero.

if respuestas_posibles > 0:

    porcentaje_disponibilidad = (
        total_respuestas_reales
        / respuestas_posibles
    ) * 100

else:

    porcentaje_disponibilidad = 0.0

# Comprobamos que el total coincida con
# los conteos individuales de los estudiantes.

if total_respuestas_reales != int(
    datos_reales["respuestas_disponibles"].sum()
):
    raise ValueError(
        "Los conteos de respuestas no coinciden."
    )

# Mostramos los resultados generales.

print("DISPONIBILIDAD TOTAL - EMOTIONLAB")

print("Respuestas disponibles:", total_respuestas_reales)
print("Respuestas posibles:", respuestas_posibles)
print("Respuestas faltantes:", respuestas_faltantes)

print(
    f"Disponibilidad general: "
    f"{porcentaje_disponibilidad:.2f}%"
)

print("\nAnálisis de integridad completado.")


DISPONIBILIDAD TOTAL - EMOTIONLAB
Respuestas disponibles: 196
Respuestas posibles: 324
Respuestas faltantes: 128
Disponibilidad general: 60.49%

Análisis de integridad completado.


## Conclusión de la Etapa 2: Preparación y validación de los datos

Durante esta etapa organizamos y validamos el dataset de EmotionLab, verificando su estructura, procedencia e integridad antes de continuar con los análisis estadísticos y el desarrollo del modelo de Machine Learning.

### 1. Preparación del dataset

Trabajamos con un total de **600 registros**, distribuidos en:

- **27 registros reales**, obtenidos de estudiantes.
- **573 registros sintéticos**, utilizados para pruebas y exploraciones.
- **12 variables**, correspondientes a los cuestionarios de las tres fases.

También revisamos los identificadores, la trazabilidad de los datos sintéticos y la escala de respuestas de 1 a 9.

### 2. Integridad de las respuestas reales

Al revisar los 27 registros reales, identificamos:

- **13 estudiantes (48,15 %)** con cuestionarios completos.
- **9 estudiantes (33,33 %)** con cuestionarios parciales.
- **5 estudiantes (18,52 %)** sin respuestas registradas.

En total, contamos con **196 respuestas disponibles de 324 posibles**, lo que representa una disponibilidad del **60,49 %**.

### 3. Disponibilidad por fase

La cantidad de formularios completos disminuyó a medida que avanzaron las actividades:

- **Fase inicial:** 21 formularios completos.
- **Fase fácil:** 14 formularios completos.
- **Fase difícil:** 13 formularios completos.

Esta disminución refleja una menor disponibilidad de información, pero no demuestra un aumento o disminución del estrés.

### 4. Consideraciones para las siguientes etapas

A partir de estos resultados, identificamos aspectos importantes:

- Mantener separados los registros reales y sintéticos.
- Considerar las respuestas ausentes antes de realizar análisis.
- Utilizar los **13 registros completos** cuando sea necesario contar con todas las variables.
- Evaluar posteriormente la posibilidad de estimar respuestas faltantes, sin confundirlas con respuestas reales.

Durante esta etapa no eliminamos registros ni reemplazamos valores ausentes.

### 5. Resultado final

La Etapa 2 nos permitió preparar una base de datos organizada y conocer sus principales limitaciones.

Con esta información podremos avanzar hacia la **Etapa 3: Análisis Exploratorio de Datos**, donde estudiaremos la distribución de las respuestas, las relaciones entre variables y las diferencias observadas entre las tres fases de EmotionLab.

**En conclusión**, logramos establecer una base de trabajo más confiable y transparente, identificando claramente qué información está disponible y qué precauciones debemos considerar antes de desarrollar nuestro modelo de Machine Learning.